In [1]:
import pandas as pd

# Carregando os dados
orders = pd.read_csv('../data/olist_orders_dataset.csv')
items = pd.read_csv('../data/olist_order_items_dataset.csv')
customers = pd.read_csv('../data/olist_customers_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')

# Checando o tamanho de cada tabela
for nome, df in [('Orders', orders), ('Items', items), ('Customers', customers),
                 ('Payments', payments), ('Reviews', reviews), ('Products', products)]:
    print(f'{nome}: {df.shape[0]:,} linhas | {df.shape[1]} colunas')

Orders: 99,441 linhas | 8 colunas
Items: 112,650 linhas | 7 colunas
Customers: 99,441 linhas | 5 colunas
Payments: 103,886 linhas | 5 colunas
Reviews: 99,224 linhas | 7 colunas
Products: 32,951 linhas | 9 colunas


In [2]:
# Verificando datas e nulos nas tabelas principais
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# Período dos dados
print(f"Período: {orders['order_purchase_timestamp'].min().date()} até {orders['order_purchase_timestamp'].max().date()}")
print(f"Total de pedidos: {len(orders):,}")
print()

# Status dos pedidos
print("Status dos pedidos:")
print(orders['order_status'].value_counts())
print()

# Nulos nas colunas de data de entrega
print("Pedidos sem data de entrega real:", orders['order_delivered_customer_date'].isna().sum())

Período: 2016-09-04 até 2018-10-17
Total de pedidos: 99,441

Status dos pedidos:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Pedidos sem data de entrega real: 2965


In [3]:
# Juntando todas as tabelas em uma só
df = orders.merge(items, on='order_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(payments.groupby('order_id').agg(
    valor_total=('payment_value', 'sum'),
    parcelas=('payment_installments', 'max'),
    forma_pagamento=('payment_type', 'first')
).reset_index(), on='order_id', how='left')
df = df.merge(reviews[['order_id', 'review_score']].drop_duplicates('order_id'), on='order_id', how='left')

# Filtrando só pedidos entregues com data de entrega válida
df = df[df['order_status'] == 'delivered']
df = df[df['order_delivered_customer_date'].notna()]

# Criando coluna de dias de entrega e se atrasou
df['dias_entrega'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['atrasou'] = df['order_delivered_customer_date'] > df['order_estimated_delivery_date']

print(f"Tabela final: {df.shape[0]:,} linhas | {df.shape[1]} colunas")
print(f"Pedidos atrasados: {df['atrasou'].sum():,} ({df['atrasou'].mean()*100:.1f}%)")
print(f"Tempo médio de entrega: {df['dias_entrega'].mean():.1f} dias")

Tabela final: 110,189 linhas | 24 colunas
Pedidos atrasados: 8,714 (7.9%)
Tempo médio de entrega: 12.0 dias


In [4]:
# Exportando a tabela tratada
df.to_csv('../outputs/olist_tratado.csv', index=False)
print("Arquivo salvo em outputs/olist_tratado.csv")
print(f"Colunas disponíveis: {list(df.columns)}")

Arquivo salvo em outputs/olist_tratado.csv
Colunas disponíveis: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'valor_total', 'parcelas', 'forma_pagamento', 'review_score', 'dias_entrega', 'atrasou']
